# 📘 GUÍA DEL PROFESOR — Unidad 3: Deep Learning
## Clasificación de Imágenes con CNN — Fashion-MNIST
**TecNM Campus Apatzingán | Ingeniería en Sistemas Computacionales**

> 🔒 **Este notebook es de uso exclusivo del instructor.**
> Contiene el código de referencia completo y las respuestas esperadas a cada pregunta guía.
> Úsalo para verificar los entregables de los alumnos y resolver dudas durante la sesión.

---
**Antes de empezar:** activar GPU → Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4

## Paso 1 – Carga de Librerías y Datos

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 4)

print('TensorFlow:', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

CLASS_NAMES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f'Entrenamiento: {X_train.shape} — Prueba: {X_test.shape}')
print(f'Rango de valores de píxel: [{X_train.min()}, {X_train.max()}]')

In [ ]:
# Visualización de 25 imágenes de muestra
fig, axes = plt.subplots(5, 5, figsize=(10, 10))
indices = np.random.choice(len(X_train), 25, replace=False)
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[indices[i]], cmap='gray')
    ax.set_title(CLASS_NAMES[y_train[indices[i]]], fontsize=9)
    ax.axis('off')
plt.suptitle('Muestra del dataset Fashion-MNIST', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Distribución de clases
unique, counts = np.unique(y_train, return_counts=True)
plt.figure(figsize=(10, 3))
plt.bar([CLASS_NAMES[i] for i in unique], counts, color='steelblue', edgecolor='white')
plt.title('Distribución de clases en el conjunto de entrenamiento')
plt.xticks(rotation=30, ha='right')
plt.ylabel('Cantidad de imágenes')
plt.tight_layout()
plt.show()
print('Conteo por clase:', dict(zip([CLASS_NAMES[i] for i in unique], counts)))

### ✅ Respuestas esperadas — Paso 1

**1. ¿Cuál es la forma (shape) de los arreglos?**
- `X_train`: (60000, 28, 28) — 60,000 imágenes de 28×28 píxeles.
- `X_test`: (10000, 28, 28) — 10,000 imágenes de 28×28 píxeles.
- `y_train` / `y_test`: vectores 1D con las etiquetas (enteros 0–9).

**2. ¿Qué representa cada dimensión?**
- Primera dimensión: número de muestras (imágenes).
- Segunda y tercera: alto y ancho en píxeles (28 × 28).
- No hay dimensión de canal porque las imágenes son en escala de grises (1 canal implícito).

**3. ¿Las clases están distribuidas uniformemente?**
- Sí: cada una de las 10 clases tiene exactamente 6,000 imágenes en entrenamiento y 1,000 en prueba.
- Importa porque un dataset balanceado permite usar accuracy como métrica confiable. Si hubiera desbalance, sería necesario usar precision, recall o F1 por clase.

> 💡 **Nota para el profesor:** un error común es que los alumnos asuman desbalance sin verificarlo. La gráfica de distribución lo aclara inmediatamente.

## Paso 2 – Preprocesamiento de Datos

In [ ]:
# Normalización: llevar píxeles de [0,255] a [0,1]
X_train_norm = X_train / 255.0
X_test_norm  = X_test  / 255.0

# Para MLP: aplanar cada imagen de 28×28 a un vector de 784
X_train_flat = X_train_norm.reshape(-1, 784)
X_test_flat  = X_test_norm.reshape(-1, 784)

# Para CNN: añadir dimensión de canal (28×28×1)
X_train_cnn = X_train_norm[..., np.newaxis]
X_test_cnn  = X_test_norm[..., np.newaxis]

# Separar 10,000 muestras de entrenamiento como validación
X_val_flat, X_train_flat_f = X_train_flat[:10000], X_train_flat[10000:]
X_val_cnn,  X_train_cnn_f  = X_train_cnn[:10000],  X_train_cnn[10000:]
y_val,       y_train_f      = y_train[:10000],       y_train[10000:]

print(f'Entrenamiento (sin val): {X_train_cnn_f.shape}')
print(f'Validación:              {X_val_cnn.shape}')
print(f'Prueba:                  {X_test_cnn.shape}')

### ✅ Respuestas esperadas — Paso 2

**1. ¿Por qué normalizar los píxeles dividiéndolos entre 255?**
- Los valores originales (0–255) tienen una escala grande que hace que el gradiente durante backpropagation sea inestable: los pesos de la red se actualizan con pasos muy grandes o muy pequeños dependiendo de la magnitud de los inputs.
- Al normalizar a [0,1] el optimizador trabaja en una escala uniforme, lo que acelera la convergencia y reduce el riesgo de que el gradiente explote o desaparezca.
- Adicionalmente, muchas funciones de activación (como sigmoid y tanh) operan mejor cerca del origen; la normalización facilita esto.

**2. ¿Diferencia entre validación y prueba?**
- **Validación**: se usa *durante* el entrenamiento para monitorear el desempeño en datos no vistos y ajustar hiperparámetros (arquitectura, learning rate, épocas). El modelo *no* aprende de estos datos, pero sí influyen en decisiones de diseño.
- **Prueba**: se usa *una sola vez* al final, para estimar el desempeño real del modelo en producción. Si se usara para tomar decisiones de diseño, se contaminaría el estimado.

**3. ¿Función de activación en la capa de salida?**
- **Softmax**, porque el problema es clasificación multiclase (10 clases mutuamente excluyentes).
- Softmax convierte el vector de salida en una distribución de probabilidad (valores entre 0 y 1 que suman 1), lo que permite interpretar la salida como 'probabilidad de pertenecer a cada clase'.
- Se usaría sigmoid solo si fuera clasificación binaria, o sigmoid independiente por clase si las clases no fueran mutuamente excluyentes (multi-label).

> 💡 **Nota para el profesor:** un error frecuente es usar sigmoid en la capa de salida para clasificación multiclase. Si el alumno lo hace, el modelo puede converger pero las salidas no suman 1 y no son probabilidades interpretables.

## Paso 3 – Modelo Base: Red Neuronal Densa (MLP)

In [ ]:
tf.random.set_seed(42)

mlp = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(784,)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(10, activation='softmax')
], name='MLP_base')

mlp.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

mlp.summary()

In [ ]:
history_mlp = mlp.fit(
    X_train_flat_f, y_train_f,
    validation_data=(X_val_flat, y_val),
    epochs=15,
    batch_size=128,
    verbose=1
)

# Curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_mlp.history['accuracy'],     label='Entrenamiento')
axes[0].plot(history_mlp.history['val_accuracy'], label='Validación')
axes[0].set_title('MLP — Accuracy')
axes[0].set_xlabel('Época'); axes[0].legend()

axes[1].plot(history_mlp.history['loss'],     label='Entrenamiento')
axes[1].plot(history_mlp.history['val_loss'], label='Validación')
axes[1].set_title('MLP — Loss')
axes[1].set_xlabel('Época'); axes[1].legend()

plt.tight_layout(); plt.show()

loss_mlp, acc_mlp = mlp.evaluate(X_test_flat, y_test, verbose=0)
print(f'MLP — Accuracy en prueba: {acc_mlp:.4f} | Loss: {loss_mlp:.4f}')

### ✅ Respuestas esperadas — Paso 3

**1. ¿Cuántos parámetros entrenables tiene el MLP?**
- Capa Dense(256): 784×256 + 256 = **200,960 parámetros**
- Capa Dense(128): 256×128 + 128 = **32,896 parámetros**
- Capa Dense(10):  128×10  + 10  = **1,290 parámetros**
- **Total: ~235,146 parámetros** (varía según la arquitectura elegida por el alumno).
- Se verifica con `model.summary()` o `model.count_params()`.

**2. ¿Las curvas de entrenamiento y validación son similares?**
- Resultado esperado: accuracy en prueba ~88–89% para el MLP con esta arquitectura.
- Si la curva de validación es notablemente más baja que la de entrenamiento → **overfitting** (el modelo memoriza en lugar de generalizar). El Dropout ayuda a reducirlo.
- Si ambas curvas son bajas y similares → **underfitting** (el modelo es demasiado simple o entrena pocas épocas).

**3. Diferencia entre accuracy y loss:**
- **Loss (sparse_categorical_crossentropy)**: función matemática diferenciable que el optimizador minimiza mediante backpropagation. Penaliza no solo la clase incorrecta sino también la *confianza* de la predicción incorrecta. Es continua y gradiente existe en todo punto.
- **Accuracy**: porcentaje de predicciones correctas. Es una métrica interpretable para humanos, pero no es diferenciable (es una función escalonada), por lo que no puede usarse como función de pérdida durante el entrenamiento.
- Analogía: loss es el 'termómetro' del optimizador; accuracy es el 'resultado' para el usuario.

> 💡 **Nota para el profesor:** es muy común que los alumnos confundan por qué no se puede optimizar directamente la accuracy. La clave es la diferenciabilidad.

## Paso 4 – Modelo Principal: Red Neuronal Convolucional (CNN)

In [ ]:
# ── CNN Configuración 1: ReLU, lr=0.001 ─────────────────────────────────────
tf.random.set_seed(42)

cnn_v1 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),

    # Bloque 1
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2,2)),

    # Bloque 2
    tf.keras.layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2,2)),

    # Clasificador
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(10, activation='softmax')
], name='CNN_v1_relu_lr001')

cnn_v1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
cnn_v1.summary()

In [ ]:
history_cnn1 = cnn_v1.fit(
    X_train_cnn_f, y_train_f,
    validation_data=(X_val_cnn, y_val),
    epochs=15, batch_size=128, verbose=1
)
loss_cnn1, acc_cnn1 = cnn_v1.evaluate(X_test_cnn, y_test, verbose=0)
print(f'CNN v1 — Accuracy: {acc_cnn1:.4f} | Loss: {loss_cnn1:.4f}')

In [ ]:
# ── CNN Configuración 2: más filtros, lr=0.0005 ──────────────────────────────
tf.random.set_seed(42)

cnn_v2 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),

    tf.keras.layers.Conv2D(64,  (3,3), activation='relu', padding='same'),
    tf.keras.layers.Conv2D(64,  (3,3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(10, activation='softmax')
], name='CNN_v2_deep_lr0005')

cnn_v2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_cnn2 = cnn_v2.fit(
    X_train_cnn_f, y_train_f,
    validation_data=(X_val_cnn, y_val),
    epochs=15, batch_size=128, verbose=1
)
loss_cnn2, acc_cnn2 = cnn_v2.evaluate(X_test_cnn, y_test, verbose=0)
print(f'CNN v2 — Accuracy: {acc_cnn2:.4f} | Loss: {loss_cnn2:.4f}')

### ✅ Respuestas esperadas — Paso 4

**1. ¿Qué hace Conv2D que Dense no puede hacer?**
- Una capa densa conecta cada neurona con *todos* los píxeles de la imagen, sin tomar en cuenta su posición espacial. Trata a la imagen como un vector plano.
- Conv2D aplica filtros (kernels) que detectan patrones locales (bordes, texturas, formas) respetando la estructura 2D de la imagen. El mismo filtro se desliza por toda la imagen (**compartición de pesos**), lo que reduce enormemente el número de parámetros y permite que la red aprenda características independientemente de su posición en la imagen (**invarianza traslacional**).
- Para imágenes, esto es crítico: el contorno de un zapato en la esquina superior izquierda y el mismo contorno en el centro derecho deben activar el mismo filtro.

**2. ¿Qué ocurre con el gradiente con sigmoide vs ReLU en capas profundas?**
- **Sigmoid**: su derivada tiene un valor máximo de 0.25 (en x=0) y se acerca a 0 en los extremos. Al multiplicar gradientes a través de muchas capas (regla de la cadena), el gradiente se vuelve exponencialmente pequeño → **vanishing gradient**. Las capas iniciales aprenden muy lentamente o nada.
- **ReLU**: su derivada es 1 para x>0 y 0 para x≤0. Para valores positivos el gradiente pasa sin atenuarse, lo que permite entrenar redes profundas de manera efectiva. El único problema es la 'neurona muerta' (dead ReLU) cuando la unidad siempre recibe valores negativos; se mitiga con inicialización correcta o usando Leaky ReLU.

**3. ¿Cómo afecta el tamaño del filtro a lo que aprende la red?**
- Filtros pequeños (3×3): detectan patrones locales finos (bordes, esquinas). Son los más usados porque apilan varios de ellos para crear campo receptivo amplio con pocos parámetros.
- Filtros grandes (5×5, 7×7): capturan contexto más amplio en una sola operación, pero tienen más parámetros y pueden perder detalle fino.
- En la práctica, 3×3 apilados dominan la arquitectura moderna (VGG, ResNet) por ser más eficientes.

> 💡 **Nota para el profesor:** si un alumno usa sigmoid en capas ocultas y nota que el modelo aprende muy lento, ese es exactamente el síntoma de vanishing gradient. Es un buen momento para hacer el experimento en clase.

## Paso 5 – Comparación de Arquitecturas y Selección del Modelo

In [ ]:
# Tabla comparativa
resultados = pd.DataFrame([
    {
        'Arquitectura':       'MLP base',
        'Activación ocultas': 'ReLU',
        'Learning Rate':      0.001,
        'Accuracy prueba (%)': round(acc_mlp  * 100, 2),
        'Loss prueba':         round(loss_mlp,  4),
        'Parámetros':          mlp.count_params()
    },
    {
        'Arquitectura':       'CNN v1',
        'Activación ocultas': 'ReLU',
        'Learning Rate':      0.001,
        'Accuracy prueba (%)': round(acc_cnn1 * 100, 2),
        'Loss prueba':         round(loss_cnn1, 4),
        'Parámetros':          cnn_v1.count_params()
    },
    {
        'Arquitectura':       'CNN v2 (deep)',
        'Activación ocultas': 'ReLU',
        'Learning Rate':      0.0005,
        'Accuracy prueba (%)': round(acc_cnn2 * 100, 2),
        'Loss prueba':         round(loss_cnn2, 4),
        'Parámetros':          cnn_v2.count_params()
    },
])

print(resultados.to_string(index=False))

# Curvas comparativas de validación
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_mlp.history['val_accuracy'],  label='MLP',    linestyle='--')
plt.plot(history_cnn1.history['val_accuracy'], label='CNN v1')
plt.plot(history_cnn2.history['val_accuracy'], label='CNN v2')
plt.title('Accuracy en Validación — comparación'); plt.legend(); plt.xlabel('Época')

plt.subplot(1, 2, 2)
plt.plot(history_mlp.history['val_loss'],  label='MLP',    linestyle='--')
plt.plot(history_cnn1.history['val_loss'], label='CNN v1')
plt.plot(history_cnn2.history['val_loss'], label='CNN v2')
plt.title('Loss en Validación — comparación'); plt.legend(); plt.xlabel('Época')

plt.tight_layout(); plt.show()

### ✅ Respuestas esperadas — Paso 5

**Resultados típicos esperados (pueden variar ligeramente por aleatoriedad):**
| Arquitectura | Accuracy prueba | Loss prueba |
|---|---|---|
| MLP base     | ~88–89%         | ~0.33       |
| CNN v1       | ~91–92%         | ~0.24       |
| CNN v2 deep  | ~92–93%         | ~0.21       |

**1. ¿La CNN superó al MLP?**
- Sí, en aproximadamente 3–4 puntos porcentuales de accuracy.
- Ocurre porque la CNN explota la estructura espacial de las imágenes: aprende a detectar bordes, texturas y formas locales que son invariantes a la posición, algo que el MLP no puede hacer al tratar la imagen como un vector plano.

**2. ¿El mejor en validación fue el mejor en prueba?**
- En general sí, pero pueden darse pequeñas discrepancias por la varianza del conjunto de prueba.
- Si hay una diferencia grande, podría indicar que el modelo se sobreajustó al conjunto de validación durante la búsqueda de hiperparámetros.

**3. ¿Riesgo de elegir solo por accuracy?**
- Un modelo con accuracy marginalmente mayor pero loss mucho más alta puede ser menos confiable en producción (predicciones con menor certeza).
- También hay que considerar el costo computacional: si CNN v2 tiene 3× más parámetros y solo gana 0.5% de accuracy, puede no justificarse para el uso práctico.
- En contextos reales también importa el tiempo de inferencia, el tamaño del modelo y la interpretabilidad.

> 💡 **Nota para el profesor:** un buen indicador de entendimiento es si el alumno justifica su elección más allá del número de accuracy.

## Paso 6 – Análisis de Errores

In [ ]:
# Usar el mejor modelo (CNN v2 en este caso)
mejor_modelo = cnn_v2

# Predicciones
y_pred_probs = mejor_modelo.predict(X_test_cnn, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Matriz de Confusión — mejor modelo (CNN v2)')
plt.ylabel('Etiqueta real'); plt.xlabel('Predicción')
plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()

# Reporte por clase
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Visualizar imágenes clasificadas incorrectamente
errores = np.where(y_pred != y_test)[0]
print(f'Total de errores: {len(errores)} de {len(y_test)} ({len(errores)/len(y_test)*100:.1f}%)')

muestra_errores = errores[:12]
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for i, ax in enumerate(axes.flat):
    idx = muestra_errores[i]
    ax.imshow(X_test[idx], cmap='gray')
    ax.set_title(
        f'Real: {CLASS_NAMES[y_test[idx]]}\nPred: {CLASS_NAMES[y_pred[idx]]}',
        fontsize=8, color='red'
    )
    ax.axis('off')
plt.suptitle('Ejemplos de clasificaciones incorrectas', fontsize=12)
plt.tight_layout(); plt.show()

### ✅ Respuestas esperadas — Paso 6

**1. ¿Qué clases se confunden con más frecuencia?**
- Los pares de mayor confusión típicos en Fashion-MNIST son:
  - **Shirt ↔ T-shirt/top**: visualmente similares (ambas son prendas de torso sin mangas largas).
  - **Shirt ↔ Coat**: contornos similares en escala de grises de 28×28.
  - **Pullover ↔ Coat**: formas superpuestas.
  - **Sneaker ↔ Ankle boot**: calzado con contornos parecidos.
- Tiene sentido visualmente: a 28×28 píxeles en escala de grises, la diferencia entre una camisa y una camiseta puede ser muy sutil.

**2. ¿Revela algún sesgo sistemático?**
- Si la clase 'Shirt' tiene recall bajo (muchas Shirts predichas como otra cosa), el modelo tiene dificultad específica con esa categoría, no solo errores aleatorios.
- Esto es un sesgo sistemático: hay algo en la representación aprendida que no distingue bien esa clase.

**3. ¿Qué modificación reduciría los errores?**
- **Data augmentation**: rotaciones, zoom, flips horizontales durante el entrenamiento para que el modelo vea más variaciones de cada clase.
- **Imágenes de mayor resolución**: con 28×28 se pierde mucho detalle; imágenes más grandes permitirían distinguir mejor los bordes y texturas finas.
- **Transfer learning**: usar un modelo preentrenado en imágenes reales (ResNet, EfficientNet) y hacer fine-tuning sobre Fashion-MNIST.
- **Aumento del peso de la clase Shirt** en la función de pérdida para penalizar más sus errores.

> 💡 **Nota para el profesor:** el reporte de classification_report muestra precision, recall y F1 por clase. Es una buena oportunidad para discutir cuándo accuracy general puede ser engañosa.

## Paso 7 – Reflexión Final e Insights

### ✅ Ejemplos de insights bien formulados (referencia para evaluar)

---
**Insight 1:**
- **Hallazgo:** La CNN supera al MLP en ~3–4% de accuracy con menos parámetros efectivos gracias a la compartición de pesos en las capas convolucionales.
- **Evidencia:** Tabla comparativa: MLP ~88%, CNN v1 ~91%, con el MLP teniendo ~235K parámetros vs ~430K de la CNN v1, pero la CNN v1 usa sus parámetros de forma más eficiente al reutilizar los filtros en toda la imagen.
- **Aplicación:** En una materia de Procesamiento de Imágenes, al proponer un proyecto de clasificación (cartas, documentos, placas), se justificaría empezar directamente con CNN en lugar de MLP.

---
**Insight 2:**
- **Hallazgo:** La confusión sistemática entre Shirt, T-shirt y Pullover indica que el modelo aprende siluetas generales pero tiene dificultad con diferencias texturales a 28×28 px.
- **Evidencia:** Matriz de confusión: las clases 0 (T-shirt), 4 (Coat) y 6 (Shirt) concentran la mayoría de los errores entre sí.
- **Aplicación:** Antes de desplegar un sistema de clasificación de prendas, se debería validar con imágenes más representativas y considerar resoluciones superiores o transfer learning desde modelos preentrenados en moda.

---
**Insight 3:**
- **Hallazgo:** ReLU converge más rápido que sigmoid en capas ocultas: en la prueba con sigmoid la red tardó el doble de épocas en alcanzar el mismo accuracy.
- **Evidencia:** Curvas de entrenamiento: con sigmoid la accuracy de validación después de 5 épocas era ~10% menor que con ReLU.
- **Aplicación:** Al diseñar redes neuronales en clase, establecer ReLU como activación predeterminada para capas ocultas y reservar sigmoid solo para capas de salida en clasificación binaria.

---
> 💡 **Criterio de evaluación sugerido para los insights del alumno:**
> - ✅ El hallazgo es específico (tiene números, no solo 'mejoró').
> - ✅ La evidencia referencia una gráfica o tabla concreta.
> - ✅ La aplicación conecta con un contexto real o docente, no es genérica.